### Import dependencies

In [ ]:
import openai
import os
import json
from qdrant_client import QdrantClient
from qdrant_client.models import Filter, FieldCondition, MatchValue
from langsmith import Client

In [26]:
from dotenv import load_dotenv
import os

load_dotenv("../../.env")

True

### Download all data from Qdrant

In [3]:
qdrant_client = QdrantClient(url='http://localhost:6333')

In [4]:
all_points = qdrant_client.scroll(
    collection_name='amazon-items-collection-01',
    limit=100,
    offset=None,
    with_payload=True,
    with_vectors=False,
)

In [5]:
all_context = [{'id':data.payload['parent_asin'], 'text':data.payload['processed_description']} for data in all_points[0]]

### Render a prompt to generate synthetic eval reference dataset

In [ ]:
output_schema ={
    'type': 'array',
    'items':{
        'type': 'object',
        'properties':{
            'reasoning':{
                'type': 'string',
                'description': 'Reasoning why the question could be answered with the given chunks.',
            },
            'question':{
                'type': 'string',
                'description':'Suggested question.',
            },
            'chunk_ids':{
                'type': 'array',
                'items':{
                    'type': 'string',
                    'description': 'ID of the chunk that could be used to answer the question.',
                },
            },
            'answer_example':{
                'type': 'string',
                'description': 'Suggested answer grounded in the context.',
            }
        },
    }
}


SYSTEM_PROMPT = f"""
I am building a RAG application. I have a collection of 50 chunks of text.
The RAG application will act as shopping assistant that can answer questions about the stock of the products we have available.
I will provide all of the available products to you with IDs of each chunk
Come up with 30 questions to which the answers could be grounded in the chunk context.
The questions should imitate a potential real user of this RAG system - a customer of the e-shop.
As an output I need you to provide me the list of questions and the IDs of the chunks that could be used to answer them.
Also provide an example answer to the question given the context of the chunks
Also provide the reason why you chose the chunks to answer the question.
Construct 10 questions that could use multiple chunks in the answer.
Construct 15 questions that could use single chunk in the answer.
Construct 5 questions that could use no chunks at all.
Do not use word chunks in suggested questions, refer to the chunks as products.

<OUTPUT_JSON_SCHEMA>
{json.dumps(output_schema, indent=2)}
</OUTPUT_JSON_SCHEMA>

I need to be able to parse the json output.
"""

USER_PROMPT = f"""
Here is the list opf chunks, each list element is a dictionary with id and text:
{json.dumps(all_context, indent=2)}
"""

In [11]:
response = openai.chat.completions.create(
    model='gpt-5.4',
    messages=[
        {'role': 'system', 'content': SYSTEM_PROMPT},
        {'role': 'user', 'content': USER_PROMPT},
    ],
    reasoning_effort='none'
)

In [12]:
print(response.choices[0].message.content)

[
  {
    "reasoning": "This question compares two cooling products. One product is a USB-powered fan for routers/modems with speed control, and the other is a laptop cooling pad with six fans and adjustable height. Both products contain cooling-specific details needed for a grounded comparison.",
    "question": "I need something to keep my electronics cool. Which product is better for a router or TV box, and which one is better for a laptop?",
    "chunk_ids": ["B0BRJS644Z", "B099N9F3FP"],
    "answer_example": "For a router or TV box, the Marame USB powered fan would be the better choice because it is specifically described for cooling routers, modems, TV boxes, receivers, and similar electronics. For a laptop, the AICHESON cooling pad is the better option because it is designed for 11-inch to 15.6-inch laptops, has 6 cooling fans, and also offers ergonomic height adjustment.",
    "reasoning_type": "multi"
  },
  {
    "reasoning": "This asks for comparison between two earbud produ

In [18]:
json_output = response.choices[0].message.content
json_output = json.loads(json_output)

### Remove reasoning_type key from objects inside JSON

In [21]:
for obj in json_output:
    obj.pop('reasoning_type')

In [23]:
single_chunk_questions = [obj for obj in json_output if len(obj['chunk_ids']) == 1]
multiple_chunk_questions = [obj for obj in json_output if len(obj['chunk_ids']) > 1]
no_chunk_questions = [obj for obj in json_output if len(obj['chunk_ids']) == 0]

print(f"Single chunk questions: {len(single_chunk_questions)}")
print(f"Multiple chunk questions: {len(multiple_chunk_questions)}")
print(f"No chunk questions: {len(no_chunk_questions)}")


Single chunk questions: 16
Multiple chunk questions: 10
No chunk questions: 5


### Fetch the item descriptions based on the chunk IDs

In [36]:
def get_descriptions(parent_asin: str) -> str:
    points = qdrant_client.scroll(
        collection_name='amazon-items-collection-01',
        scroll_filter=Filter(
            must=[
                FieldCondition(key='parent_asin', match=MatchValue(value=parent_asin))
            ]
        ),
    )[0]

    return points[0].payload['processed_description']

### Create Eval dataset in Langsmith

In [25]:
ls_client = Client()

In [27]:
dataset_name = 'rag-eval-dataset'
dataset = ls_client.create_dataset(
    dataset_name=dataset_name,
    description='Eval dataset for RAG system',
    metadata={
        'project_name': 'e2e_ai_engineering_bootcamp',
    }
)

In [37]:
for item in json_output:
    ls_client.create_example(
        dataset_id=dataset.id,
        inputs={
            'question': item['question'],
        },
        outputs={
            'ground_truth': item['answer_example'],
            'retrieved_context_ids': item['chunk_ids'],
            'reference_descriptions': [get_descriptions(id) for id in item['chunk_ids']],
        },
    )